# Lecția 12 - Reducerea Istoricului Conversațiilor cu Agent Scratchpad

Acest notebook demonstrează cum să gestionezi contextul în conversații lungi folosind Microsoft Agent Framework. Pe măsură ce conversațiile cresc, numărul de tokeni crește — depășind în cele din urmă fereastra de context a modelului. Abordăm această problemă cu un **pattern de sumarizare a contextului** și un **agent scratchpad** pentru memorie persistentă.

## Ce vei învăța:
1. **De ce contează gestionarea contextului**: Înțelegerea limitelor de tokeni și a ferestrelor de context
2. **Agenți conștienți de context**: Construirea unor agenți care își gestionează propriul context al conversației
3. **Pattern de sumarizare a contextului**: Folosirea unor unelte pentru a condensa istoricul conversației
4. **Agent Scratchpad**: Memorie persistentă care supraviețuiește reducerii contextului

## Cerințe preliminare:
- Configurarea Azure OpenAI cu variabile de mediu setate
- Înțelegerea conceptelor de bază ale agenților din lecțiile anterioare


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## De ce este importantă gestionarea contextului

Fiecare LLM are o **fereastră de context** finită — numărul maxim de tokenuri pe care le poate procesa într-o singură cerere. Pe măsură ce o conversație multi-turn avansează:

- **Numărul de tokenuri crește liniar** cu fiecare mesaj al utilizatorului și răspuns al asistentului.
- **Tokenurile din prompt domină costul** deoarece întreg istoricul este retrimis la fiecare tură.
- În cele din urmă, conversația **depășește fereastra de context** și modelul fie trunchiază, fie generează o eroare.

### Strategii pentru gestionarea contextului

| Strategie | Cum funcționează | Compromisuri |
|---|---|---|
| **Trunchiere** | Se elimină cele mai vechi mesaje | Se pierde contextul inițial |
| **Rezumat** | Mesajele mai vechi se condensă într-un rezumat | Se pierd unele detalii, dar punctele cheie sunt păstrate |
| **Blocnotes / Memorie externă** | Se stochează fapte cheie în afara conversației | Necesită apeluri către unelte, dar supraviețuiește oricărei reduceri |

În acest notebook combinăm **rezumatul** cu o **unealtă blocnotes** astfel încât agentul să poată menține continuitatea chiar și atunci când istoricul conversației este condensat.


## Crearea unui Agent Conștient de Context


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simularea unei conversații lungi

Să parcurgem o conversație cu mai multe schimburi pentru a vedea cum se acumulează contextul. Agentul ar trebui să rețină detalii cheie (preferințe, buget, date de călătorie) pe parcursul schimburilor și să demonstreze continuitate.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Observați cum agentul păstrează contextul din tura anterioară — știe despre Japonia, sushi, temple, fotografie, bugetul de 3000 de dolari, călătoria solo și perioada din aprilie. Într-o conversație scurtă, acest lucru funcționează bine, dar pe măsură ce conversația crește, istoricul complet devine costisitor de re-expediat.

Să continuăm conversația cu mai multe ture pentru a vedea acumularea contextului:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Model pentru Rezumatul Contextului

Pe măsură ce conversația crește, putem folosi un **instrument de rezumat** pentru a comprima contextul acumulat într-un format compact. Agentul apelează acest instrument pentru a înregistra preferințele cheie, astfel încât, chiar dacă mesajele mai vechi sunt eliminate, informațiile esențiale sunt păstrate.

Acest model este piatra de temelie pentru o reducere mai sofisticată a istoricului:
1. Agentul identifică faptele cheie din conversație
2. Apeleză instrumentul de rezumat pentru a le păstra
3. Mesajele mai vechi pot fi eliminate în siguranță deoarece rezumatul capturează ceea ce contează

Mai jos definim un instrument `summarize_preferences` pe care agentul îl poate apela pentru a înregistra un rezumat compact al ceea ce a învățat.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Rezumat

În această lecție ai învățat cum să gestionezi contextul în conversații cu agenți pe termen lung folosind Microsoft Agent Framework:

### Concepte cheie
- **Ferestrele de context sunt finite** — fiecare token din istoricul conversației costă bani și contribuie la limită.
- **Instrumentele de rezumare** permit agentului să condenseze contextul acumulat în rezumate compacte, reducând utilizarea tokenilor păstrând informațiile esențiale.
- **Blocnotesurile agentului** oferă memorie externă persistentă care supraviețuiește oricărei reduceri a conversației.

### Ce ai construit
- Un **agent conștient de context** care menține continuitatea în conversații cu mai multe tururi
- Un **instrument de rezumare** (`summarize_preferences`) care înregistrează detalii-cheie despre utilizator într-un format compact
- O **conversație multi-tur** demonstrând reținerea contextului și gestionarea schimbărilor

### Aplicații în lumea reală
- **Boți pentru servicii clienți**: Reamintesc preferințele pe durata unor sesiuni lungi de suport
- **Asistenți personali**: Urmăresc proiecte în desfășurare fără a necesita re-explicarea contextului
- **Profesori educaționali**: Mențin progresul elevului pe parcursul a numeroase interacțiuni

### Pașii următori
- Implementează un instrument complet de blocnotes cu persistență bazată pe fișiere
- Adaugă trunchiere automată a istoricului după rezumare
- Combină cu baze de date vectoriale pentru căutare semantică în memorie
- Construiește agenți care pot relua conversațiile zile mai târziu cu context complet


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
